# 1.10 — Calling Class Methods 🧋

### AP CSA · Unit 1: Using Objects and Methods
**Boba Cafe Series · Lesson 10**

---

> **Setup note:** every code cell runs on the **IJava kernel** (Java 17+). Check that the kernel picker says *Java*. Each cell ends with a call to whichever class actually contains `main`.

## The word you've typed a hundred times without asking about

Look back through every notebook in this series. The keyword `static` sits in the header of literally every method you have ever written — `public static void main`, `public static double calculateTotal`, all of it. You typed it because the template told you to, and it worked, so you moved on.

Today you find out what it was actually promising.

The cafe is expanding. The manager wants the **same pricing logic** — tax, bulk discounts, loyalty math — used by both the in-store register and a brand-new mobile ordering app. Two different programs, one shared toolkit. That's not a hypothetical: it's exactly the situation `static` was built for, and it's the reason libraries like `Math` can be called from any class in any program without anyone ever creating a `Math` object.

### What you'll be able to do by the end

| # | Objective | CED reference |
|---|---|---|
| 1 | Explain what `static` means: a method that belongs to the **class**, not to any object | 1.10.A |
| 2 | Call a class method from **within** the class that defines it | 1.10.A |
| 3 | Call a class method from a **different** class using `ClassName.method(...)` | 1.10.A |
| 4 | Trace the **flow of control** as execution jumps into and back out of a method | 1.10.A |
| 5 | Diagnose common calling mistakes: missing parentheses, wrong class name, wrong arguments | 1.10.A |

---

## Part 1 — What `static` actually means

> **Definition:** A **class method** — one declared with the `static` keyword — belongs to the **class itself**, not to any particular object. You call it through the class name, and no object ever needs to be created.

Think of the difference this way. `Math` isn't a drink; it's a **supplier's catalog**. You don't build a "Math object" before asking it for a square root, the same way you don't have to construct a personal calculator before punching in a formula — you just reach for the shared one on the wall labeled `Math`.

```
   Math.sqrt(144)
   ^^^^ ^^^^^^^^^
    |       |
    |       └─ the method being called
    └─ the CLASS the method belongs to -- no object anywhere in sight
```

Every method you've written in this series so far has been `static`, which is exactly why you never had to build anything before calling it. `Math`, `Integer`, and every helper method from Lessons 1.8 and 1.9 are all class methods.

> **A preview, not today's topic:** later in this unit (Topic 1.14) you'll meet **instance methods** — methods that belong to a specific *object*, like one particular `Scanner` you built, called with `scan.nextInt()` instead of `Scanner.nextInt()`. That distinction is coming. For now, everything is a class method, and `static` is the word that says so.

In [ ]:
public class WhatStaticMeans {
    public static void main(String[] args) {
        // No object was created anywhere. Every one of these is reached
        // through its CLASS name, because every one of them is static.
        System.out.println("sqrt:      " + Math.sqrt(64));
        System.out.println("MAX_VALUE: " + Integer.MAX_VALUE);
        System.out.println("random:    " + (Math.random() < 1.0));   // always true, just proving it runs
    }
}

WhatStaticMeans.main(null);

---

## Part 2 — Calling a method from within its own class

When you call a class method from **inside the same class that defines it**, you don't need the class name at all. Just the method name.

You've been doing this since Lesson 1.8 without it being named.

In [ ]:
public class SameClassCall {

    public static double taxedPrice(double price) {
        return price + price * 0.0775;
    }

    public static void printReceipt(double price) {
        // taxedPrice is defined in THIS class, so no prefix is needed here
        System.out.println("Price with tax: $" + taxedPrice(price));
    }

    public static void main(String[] args) {
        printReceipt(5.75);      // also same-class -- no prefix
    }
}

SameClassCall.main(null);

Both calls — `taxedPrice(price)` inside `printReceipt`, and `printReceipt(5.75)` inside `main` — skip the class name entirely, because Java already knows what class you're standing in.

---

## Part 3 — Calling a method from a different class

This is the new piece, and it's the one that makes the manager's request possible: **write the pricing logic once, and let a completely different class use it.**

To call a method defined in another class, you need the class name, a dot, and the method call:

```
   ClassName.methodName(arguments)
```

This is exactly the syntax you've used with `Math.sqrt` and `Integer.MAX_VALUE` the whole time — those are simply calls into classes Oracle wrote instead of classes you wrote. The rule doesn't change based on who authored the class.

Below, `MenuMath` is the toolkit — it has no `main` and can't be run on its own. `OrderApp` is the program that uses it. Notice `OrderApp` never recreates the tax formula; it just calls into `MenuMath`.

In [ ]:
public class OrderApp {
    public static void main(String[] args) {
        // MenuMath is a DIFFERENT class -- the class name is required.
        double taxed = MenuMath.taxedPrice(5.75);
        System.out.println("Taxed price:    $" + taxed);

        double discounted = MenuMath.bulkDiscount(5.75, 6);
        System.out.println("Bulk of 6:      $" + discounted);
    }
}

class MenuMath {
    public static double taxedPrice(double price) {
        return price + price * 0.0775;
    }

    public static double bulkDiscount(double unitPrice, int quantity) {
        return quantity >= 5 ? unitPrice * 0.9 : unitPrice;
    }
}

OrderApp.main(null);

`MenuMath` never runs by itself — it has no `main` method, and it isn't meant to. It exists purely to be **called into** by other classes, the exact same role `Math` plays for every Java program on earth.

> **A note about this notebook's structure:** when two classes share one Java file, the class that contains `main` must be declared **first**. That's a quirk of how a single Java source file gets launched, not a rule about calling methods in general — in a real multi-file project each class simply lives in its own `.java` file and this ordering issue doesn't come up at all.

### Reusing the same toolkit from a second, unrelated class

This is the actual payoff. Watch `MenuMath` get called from a completely different program with a completely different purpose — nothing about `MenuMath` had to change.

In [ ]:
public class MobileApp {
    public static void main(String[] args) {
        // Same MenuMath class, same methods, a totally different caller.
        System.out.println("Mobile order taxed price: $" + MenuMath.taxedPrice(6.50));
        System.out.println("Mobile bulk order (8):    $" + MenuMath.bulkDiscount(6.50, 8));
    }
}

class MenuMath {
    public static double taxedPrice(double price) {
        return price + price * 0.0775;
    }

    public static double bulkDiscount(double unitPrice, int quantity) {
        return quantity >= 5 ? unitPrice * 0.9 : unitPrice;
    }
}

MobileApp.main(null);

`OrderApp` and `MobileApp` never saw each other's source code, and they didn't need to. Both simply called `MenuMath.taxedPrice(...)` and trusted the API — precisely the abstraction idea from Lesson 1.7, except now **you're** the one writing the library instead of just reading one.

---

## Part 4 — Tracing the flow of control

> **Flow of control** describes the order statements actually execute in — and a method call **interrupts** that order. Execution jumps into the called method, runs it completely, and then resumes exactly where it left off.

```
   main() is running...
        |
        |   sealCup();     <-- execution JUMPS into sealCup
        |        |
        |        |  (sealCup's statements run, top to bottom)
        |        |
        |   <-------------- once sealCup finishes, control RETURNS here
        |
   ...main() continues on the very next line
```

Read the trace below by watching the **indentation**, not just the text — it mirrors exactly how deep into a call the program currently is.

In [ ]:
public class FlowOfControl {

    public static void sealCup() {
        System.out.println("  sealCup: starting");
        addLabel();                                    // JUMPS into addLabel
        System.out.println("  sealCup: finishing");    // resumes here after addLabel returns
    }

    public static void addLabel() {
        System.out.println("    addLabel: starting");
        System.out.println("    addLabel: finishing");
    }

    public static void main(String[] args) {
        System.out.println("main: before the call");
        sealCup();                                      // JUMPS into sealCup
        System.out.println("main: after the call");     // resumes here after sealCup returns
    }
}

FlowOfControl.main(null);

Six statements, but they did **not** print in the order they're written on the page. `main` paused mid-execution — twice, once for each level of nesting — and picked up exactly where it left off both times. This is why a call is sometimes described as a **detour**: control leaves, does the whole side trip, and comes back to the very next line.

This matters immediately once a method returns a value: the value isn't available until the *entire* detour finishes.

In [ ]:
public class ValueAfterDetour {

    public static double taxedPrice(double price) {
        System.out.println("  (inside taxedPrice, computing...)");
        return price + price * 0.0775;
    }

    public static void main(String[] args) {
        System.out.println("main: about to call taxedPrice");
        double result = taxedPrice(5.75);          // main WAITS for this whole detour
        System.out.println("main: got back " + result);
    }
}

ValueAfterDetour.main(null);

The line `double result = taxedPrice(5.75);` doesn't finish — and `result` doesn't get its value — until every statement inside `taxedPrice` has run and a `return` has fired. Assignment (from Lesson 1.4) was always "evaluate the whole right side first"; now you can see that "evaluating" a method call means **fully completing the detour**.

---

## Part 5 — Three ways a call goes wrong

### Mistake 1: forgetting the parentheses

A method's signature always includes parentheses, even with zero parameters (Lesson 1.9). Leaving them off doesn't call the method — it tries to treat the method's *name* as if it were a variable, which makes no sense to the compiler.

**Heads up: the next cell is supposed to fail.**

In [ ]:
public class ForgottenParens {

    public static void seal() {
        System.out.println("Cup sealed.");
    }

    public static void main(String[] args) {
        seal;      // missing ()  -- this does not call the method
    }
}

ForgottenParens.main(null);

`error: not a statement`. Java doesn't recognize `seal` on its own as anything meaningful — a method name without parentheses isn't a call, it's just a dangling word. The fix is `seal();`.

### Mistake 2: the wrong class name

**Heads up: the next cell is supposed to fail.**

In [ ]:
public class WrongClassName {
    public static void main(String[] args) {
        double p = MenuMathd.taxedPrice(5.75);      // typo: MenuMathd, not MenuMath
        System.out.println(p);
    }
}

class MenuMath {
    public static double taxedPrice(double price) {
        return price + price * 0.0775;
    }
}

WrongClassName.main(null);

`cannot find symbol / variable MenuMathd`. Java has no idea what `MenuMathd` refers to, because nothing by that name was ever declared. This is a close cousin of the "cannot find symbol" error from Lesson 1.7, which came from a missing `import` — here the class exists, but the name typed at the call site doesn't match it.

### Mistake 3: wrong number or type of arguments

This is exactly the signature-matching rule from Lesson 1.9, showing up again as a calling mistake rather than a definition mistake.

In [ ]:
public class WrongArgs {
    public static void main(String[] args) {
        // bulkDiscount needs (double, int) -- this passes only one argument
        double d = MenuMath.bulkDiscount(5.75);
        System.out.println(d);
    }
}

class MenuMath {
    public static double bulkDiscount(double unitPrice, int quantity) {
        return quantity >= 5 ? unitPrice * 0.9 : unitPrice;
    }
}

WrongArgs.main(null);

No overload of `bulkDiscount` accepts a single argument, so nothing matches and the call fails to compile. **All three mistakes here are compile-time errors** — Java refuses to even build the program rather than let a broken call run. That's worth appreciating: it's the run-time and logic errors from earlier lessons that hide, not this kind.

---

# Practice: Building the Shared Toolkit

Four tasks, in order.

---

## Hack 1 — Same class or different class?

For each call, say whether it needs a class-name prefix, and if so, which class. Double-click to edit.

```java
public class Register {
    public static double ringUp(double price, int qty) {
        double sub = price * qty;
        double tax = TaxTools.applyTax(sub);      // call A
        return round(tax);                        // call B
    }

    public static double round(double amount) {
        return (int)(amount * 100 + 0.5) / 100.0;
    }

    public static void main(String[] args) {
        double total = ringUp(5.75, 3);            // call C
        System.out.println(TaxTools.rate());       // call D
    }
}
```

| Call | Same class or different? | Prefix needed? |
|---|---|---|
| A — `TaxTools.applyTax(sub)` | | |
| B — `round(tax)` | | |
| C — `ringUp(5.75, 3)` | | |
| D — `TaxTools.rate()` | | |

<details>
<summary><b>Check your answers</b></summary>

| Call | Same class or different? | Prefix needed? |
|---|---|---|
| A | Different — `TaxTools` | Yes, `TaxTools.` |
| B | Same — `round` is defined in `Register` | No |
| C | Same — `ringUp` is defined in `Register`, called from `main`, also in `Register` | No |
| D | Different — `TaxTools` | Yes, `TaxTools.` |

The rule is entirely about **where the method is defined**, never about where you happen to be calling it from in terms of which method you're inside.
</details>

---

## Hack 2 — Trace the flow of control

Predict the exact printed order **before** running this cell. Double-click to write your prediction.

```java
public static void main(String[] args) {
    System.out.println("1");
    stepA();
    System.out.println("6");
}
public static void stepA() {
    System.out.println("2");
    stepB();
    System.out.println("5");
}
public static void stepB() {
    System.out.println("3");
    System.out.println("4");
}
```

**Your prediction (just the order of numbers):**

In [ ]:
public class TraceFlow {

    public static void stepB() {
        System.out.println("3");
        System.out.println("4");
    }

    public static void stepA() {
        System.out.println("2");
        stepB();
        System.out.println("5");
    }

    public static void main(String[] args) {
        System.out.println("1");
        stepA();
        System.out.println("6");
    }
}

TraceFlow.main(null);

<details>
<summary><b>Check your answer</b></summary>

`1, 2, 3, 4, 5, 6` — in that exact order.

Nothing gets skipped and nothing runs out of sequence; every call is a complete round trip. `main` prints `1`, then fully detours through `stepA`, which itself prints `2`, fully detours through `stepB` (printing `3` and `4`), returns to finish `stepA` by printing `5`, and only then does control return to `main` to print `6`.

The number of "levels deep" you are — `main` → `stepA` → `stepB` — is sometimes called the **call stack**. Each method waits for everything it called to fully finish before it can finish itself.
</details>

---

## Hack 3 — Fix the mobile app

The cell below has **two** bugs from Part 5. Run it, fix both, then fill in the report.

The correct output should be:

```
Taxed:    $6.195625
Sealed.
Bulk (6): $5.175
```

In [ ]:
public class BrokenMobileApp {
    public static void main(String[] args) {
        double taxed = PriceTools.taxedPrice(5.75);
        System.out.println("Taxed:    $" + taxed);

        seal;

        double bulk = PriceToolz.bulkDiscount(5.75, 6);
        System.out.println("Bulk (6): $" + bulk);
    }

    public static void seal() {
        System.out.println("Sealed.");
    }
}

class PriceTools {
    public static double taxedPrice(double price) {
        return price + price * 0.0775;
    }
    public static double bulkDiscount(double unitPrice, int quantity) {
        return quantity >= 5 ? unitPrice * 0.9 : unitPrice;
    }
}

BrokenMobileApp.main(null);

**Your bug report** (double-click to edit):

| Bug | What was wrong | Your fix |
|---|---|---|
| 1 | | |
| 2 | | |

<details>
<summary><b>Check your answers</b></summary>

**Bug 1 — missing parentheses.** `seal;` isn't a call. `seal` is also defined in `BrokenMobileApp` itself, so it doesn't even need a class prefix — just parentheses:

```java
seal();
```

**Bug 2 — wrong class name.** `PriceToolz` doesn't exist; the class is `PriceTools`. Fix the typo:

```java
double bulk = PriceTools.bulkDiscount(5.75, 6);
```

`PriceTools.taxedPrice(5.75)` on the first line was already correct — that one wasn't a bug.
</details>

---

## Hack 4 — Build your own two-class toolkit

Design a class called `InventoryTools` with **three** static methods any cafe program could call:

1. `fullTrays(int cups, int cupsPerTray)` — returns how many complete trays the cups fill
2. `isLowStock(int cupsRemaining, int threshold)` — returns `true` if stock is at or below the threshold
3. `restockAmount(int cupsRemaining, int targetStock)` — returns how many more cups are needed to reach the target (never a negative number)

Then write a **separate** class called `StockroomApp` with a `main` method that calls all three from `InventoryTools`, using the class-name prefix each time.

Remember: whichever class contains `main` must be declared **first** in the cell.

In [ ]:
public class StockroomApp {
    public static void main(String[] args) {

        // TODO: call all three InventoryTools methods and print each result
        //       try cups = 47, cupsPerTray = 6, threshold = 10, targetStock = 100

    }
}

class InventoryTools {

    // TODO 1: fullTrays(int cups, int cupsPerTray)

    // TODO 2: isLowStock(int cupsRemaining, int threshold)

    // TODO 3: restockAmount(int cupsRemaining, int targetStock)

}

StockroomApp.main(null);

<details>
<summary><b>One possible solution</b></summary>

```java
public class StockroomApp {
    public static void main(String[] args) {
        int cups = 47;
        int cupsPerTray = 6;
        int threshold = 10;
        int targetStock = 100;

        System.out.println("Full trays:     " + InventoryTools.fullTrays(cups, cupsPerTray));
        System.out.println("Low stock?      " + InventoryTools.isLowStock(cups, threshold));
        System.out.println("Restock needed: " + InventoryTools.restockAmount(cups, targetStock));
    }
}

class InventoryTools {

    public static int fullTrays(int cups, int cupsPerTray) {
        return cups / cupsPerTray;
    }

    public static boolean isLowStock(int cupsRemaining, int threshold) {
        return cupsRemaining <= threshold;
    }

    public static int restockAmount(int cupsRemaining, int targetStock) {
        int needed = targetStock - cupsRemaining;
        return needed > 0 ? needed : 0;
    }
}

StockroomApp.main(null);
```

Output:

```
Full trays:     7
Low stock?      false
Restock needed: 53
```

`StockroomApp` never had to know *how* `InventoryTools` computes any of these — only what to call and what comes back. That's the same abstraction from Lesson 1.7, except this time both sides of it are yours.
</details>

---

# Self-Check: AP-style questions

**1.** What does the `static` keyword in a method header indicate?

&nbsp;&nbsp;(A) The method cannot be called more than once
&nbsp;&nbsp;(B) The method belongs to the class itself, not to any object
&nbsp;&nbsp;(C) The method's return value never changes
&nbsp;&nbsp;(D) The method must be called before `main`

<details><summary>Answer</summary>

**(B)**. A `static` method — a class method — is called through the class name, with no object required. (A) and (C) describe nothing about `static` at all.
</details>

---

**2.** A method `computeFee` is defined in class `Billing`. Which call is correct from **inside** the `Billing` class itself?

&nbsp;&nbsp;(A) `Billing.computeFee(x);`
&nbsp;&nbsp;(B) `computeFee(x);`
&nbsp;&nbsp;(C) `this.computeFee(x);`
&nbsp;&nbsp;(D) `Billing->computeFee(x);`

<details><summary>Answer</summary>

**(B)**. From inside the same class that defines it, no class-name prefix is needed. (A) would also technically work but isn't required; the point is that the plain call is sufficient and standard from within the class. (D) isn't Java syntax at all.
</details>

---

**3.** What is printed?

```java
public static void main(String[] args) {
    System.out.println("A");
    helper();
    System.out.println("D");
}
public static void helper() {
    System.out.println("B");
    System.out.println("C");
}
```

&nbsp;&nbsp;(A) `A B C D` &nbsp;&nbsp; (B) `A D B C` &nbsp;&nbsp; (C) `B C A D` &nbsp;&nbsp; (D) `A B D C`

<details><summary>Answer</summary>

**(A)**. `main` prints `A`, then fully detours into `helper`, which prints `B` and `C` in order, and only after `helper` completely finishes does control return to `main` to print `D`.
</details>

---

**4.** A class `Tools` (with no `main` method) defines `public static int square(int x)`. Which statement correctly calls it from a different class named `App`?

&nbsp;&nbsp;(A) `int y = square(5);`
&nbsp;&nbsp;(B) `int y = Tools.square(5);`
&nbsp;&nbsp;(C) `int y = App.square(5);`
&nbsp;&nbsp;(D) `int y = Tools(5).square;`

<details><summary>Answer</summary>

**(B)**. `square` is defined in `Tools`, and it's being called from a different class, `App`, so the `Tools.` prefix is required. (A) would only work from inside `Tools` itself. (C) names the wrong class. (D) isn't valid syntax.
</details>

---

**5.** A method `seal()` takes no parameters. What happens when a program calls `seal;` instead of `seal();`?

&nbsp;&nbsp;(A) It calls the method normally; the parentheses are optional for no-argument methods
&nbsp;&nbsp;(B) A compile-time error, because `seal` alone is not a valid statement
&nbsp;&nbsp;(C) A run-time error when the line executes
&nbsp;&nbsp;(D) It calls the method but ignores its return value

<details><summary>Answer</summary>

**(B)**. A method signature always includes parentheses, even with an empty parameter list. Omitting them doesn't invoke the method at all — the compiler reports "not a statement" and refuses to build the program.
</details>

---

**6.** Which best explains why `Math.sqrt(16)` doesn't require creating a `Math` object first?

&nbsp;&nbsp;(A) `Math` has no constructor defined anywhere in Java
&nbsp;&nbsp;(B) `sqrt` is a static method, so it belongs to the `Math` class itself rather than to any object
&nbsp;&nbsp;(C) `sqrt` is a special built-in keyword, not a real method
&nbsp;&nbsp;(D) Java automatically creates a hidden `Math` object the first time it's used

<details><summary>Answer</summary>

**(B)**. `sqrt` is declared `static`, meaning it belongs to the class rather than to any instance. That's exactly why it's called as `Math.sqrt(...)` with no object anywhere in sight.
</details>

---

**7.** A method `bulkDiscount(double unitPrice, int quantity)` is called as `MenuMath.bulkDiscount(5.75)`. What happens?

&nbsp;&nbsp;(A) `quantity` defaults to `0`
&nbsp;&nbsp;(B) A compile-time error, because the argument count doesn't match the parameter list
&nbsp;&nbsp;(C) A run-time error when `quantity` is used
&nbsp;&nbsp;(D) It runs correctly, since `quantity` isn't used until later in the method

<details><summary>Answer</summary>

**(B)**. Java has no concept of default or optional parameters. Every parameter in the header requires exactly one matching argument at the call site, or the call fails to compile entirely — regardless of whether the missing parameter would have mattered inside the method body.
</details>

---

# Closing time

### Vocabulary to know cold

| Term | One-line definition |
|---|---|
| Class method | A `static` method — belongs to the class, callable with no object |
| `ClassName.method(...)` | The syntax for calling a method defined in a **different** class |
| Same-class call | Calling a method with no prefix, from inside the class that defines it |
| Flow of control | The order statements actually execute in, including detours through method calls |
| Call stack | The nested chain of "waiting to finish" methods during a call |

### The five things that will show up on the exam

1. `static` means the method belongs to the **class**, not an object.
2. Inside the defining class: call with **just the name**. From elsewhere: `ClassName.method(...)`.
3. A call is a **detour** — control leaves, runs the whole method, and returns to the very next line.
4. A returned value isn't available until the **entire** detour completes.
5. Method names always need **parentheses**, even with zero parameters.

### Before you submit, check that you:

- [ ] Ran every code cell, including the three that fail on purpose
- [ ] Classified all four calls in Hack 1 as same-class or different-class
- [ ] Predicted the Hack 2 output order **before** running it
- [ ] Fixed both bugs in Hack 3
- [ ] Built a working `InventoryTools` / `StockroomApp` pair in Hack 4
- [ ] Attempted all seven self-check questions before revealing answers

### Next up

**1.11 — Math Class.** You've used `Math.sqrt`, `Math.pow`, `Math.abs`, and `Integer.MAX_VALUE` for lessons now, one method at a time as each example needed it. Next shift is the payoff: the full `Math` class laid out at once, including `Math.random()` — which is the tool that finally lets Byte-Sized Boba simulate an unpredictable rush of customers instead of a scripted one.

See you at the next shift 🧋